# MLflow Example: Tracking Two Model Training Runs

In [1]:
%pip install -q scikit-learn

In [2]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [3]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [4]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [5]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...


Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [6]:
%pip install -q mlflow

In [7]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [8]:
%pip install -q dagshub

In [9]:
# get this code from dagshub (Remote -> Experiments)
# Configure le tracking URI MLflow vers https://dagshub.com/liloubkr/mlflow-with-daghub.mlflow
import dagshub
dagshub.init(repo_owner='liloubkr', repo_name='mlflow-with-daghub', mlflow=True)

print("Tracking URI:", mlflow.get_tracking_uri())

Accessing as liloubkr

Initialized MLflow to track repo "liloubkr/mlflow-with-daghub"

Repository liloubkr/mlflow-with-daghub initialized!

Tracking URI: https://dagshub.com/liloubkr/mlflow-with-daghub.mlflow


In [10]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [11]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/185138b3e8184635b68470dc084cc8ec', creation_time=1794843917239, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1794843917239, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [12]:
import os
import tempfile

# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name) as run: # <--
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)

        # Le serveur MLflow de DagsHub est en version 2.x et ne gère pas les "logged models" de MLflow 3.
        # On sauvegarde donc le modèle en local puis on l'envoie comme artefact classique du run (dossier "model").
        # skops_trusted_types : MLflow 3 sauvegarde au format sécurisé "skops", on déclare le type des arbres comme fiable.
        with tempfile.TemporaryDirectory() as tmp:
            local_path = os.path.join(tmp, "model")
            mlflow.sklearn.save_model(
                model,
                local_path,
                input_example=X_train[:5],
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            )
            mlflow.log_artifacts(local_path, artifact_path="model")

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")
        return run.info.run_id, accuracy


In [13]:
# Train and log Model 1
print("Training and Logging Model 1...")
run_id_1, acc_1 = train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

Training and Logging Model 1...


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389


🏃 View run random_forest_model_run_1 at: https://dagshub.com/liloubkr/mlflow-with-daghub.mlflow/#/experiments/0/runs/747398116b2a444e998ccabd54982878
🧪 View experiment at: https://dagshub.com/liloubkr/mlflow-with-daghub.mlflow/#/experiments/0


In [14]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
run_id_2, acc_2 = train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_2")


Training and Logging Model 2...


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722


🏃 View run random_forest_model_run_2 at: https://dagshub.com/liloubkr/mlflow-with-daghub.mlflow/#/experiments/0/runs/8380b739523243438c48a91f52483a66
🧪 View experiment at: https://dagshub.com/liloubkr/mlflow-with-daghub.mlflow/#/experiments/0


## Register the best model into the MLflow Model Registry
On enregistre le meilleur des deux runs sous le nom `RandomForest_Digits`. Chaque nouvel enregistrement crée une nouvelle version (1, 2, ...).

In [15]:
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException

model_name = "RandomForest_Digits"
client = MlflowClient()

best_run_id = run_id_1 if acc_1 >= acc_2 else run_id_2
print(f"Best run: {best_run_id} (accuracy={max(acc_1, acc_2):.4f})")

# Créer le modèle dans le registry s'il n'existe pas encore
try:
    client.create_registered_model(model_name)
except MlflowException:
    pass  # il existe déjà

# Ajouter une nouvelle version qui pointe vers le dossier "model" du meilleur run
best_run = client.get_run(best_run_id)
registered = client.create_model_version(
    name=model_name,
    source=f"{best_run.info.artifact_uri}/model",
    run_id=best_run_id,
)
print(f"Registered '{registered.name}' version {registered.version}")

Best run: 8380b739523243438c48a91f52483a66 (accuracy=0.9722)


Registered 'RandomForest_Digits' version 1


## Load a model from the MLflow Model Registry
URI de la forme `models:/<nom>/<version>` (ou `models:/<nom>@<alias>` si le serveur gère les alias).

In [16]:
# Optionnel : mettre un alias "champion" sur cette version (pas supporté par tous les serveurs)
try:
    client.set_registered_model_alias(model_name, "champion", registered.version)
    print("Alias 'champion' ->", registered.version)
except MlflowException as e:
    print("Alias non supporté par ce serveur, on charge par numéro de version.")

# Charger le modèle depuis le registry et prédire
loaded_model = mlflow.sklearn.load_model(f"models:/{model_name}/{registered.version}")
preds = loaded_model.predict(X_test)
print(f"Loaded model accuracy on test set: {accuracy_score(y_test, preds):.4f}")
print("First 10 predictions:", preds[:10], "| true:", y_test[:10])

Alias 'champion' -> 1


Loaded model accuracy on test set: 0.9722
First 10 predictions: [6 9 3 7 2 1 5 2 5 2] | true: [6 9 3 7 2 1 5 2 5 2]


In [ ]:
# Instructions to visualize results
print("Les runs et le modèle sont sur DagsHub :")
print("  https://dagshub.com/liloubkr/mlflow-with-daghub.mlflow")
print("\nPour un tracking local (sans la cellule dagshub.init), lancer dans un terminal :")
print("  mlflow ui --backend-store-uri sqlite:///mlflow.db")
print("puis ouvrir http://127.0.0.1:5000")